# 00 — Context: Verification Becomes a Resource

**Primary specification:** confidence scheduling specifies scalable LLM serving by treating target-model verification as a schedulable engineering resource.

This notebook anchors the `confidence-scheduled-decoding` repository. It turns the DSpark paper into a small executable model of speculative decoding, confidence estimation, verification scheduling, and throughput-aware serving.

The goal is not to reproduce DeepSeek's production stack in notebook 00. The goal is to make the core constraints explicit, measurable, and reusable across later notebooks.


## 1. Start here

This repository studies one engineering question:

> **How should confidence allocate verification resources during speculative decoding?**

The notebooks progress from:

\[
\text{autoregressive inference}
\rightarrow
\text{speculative decoding}
\rightarrow
\text{confidence estimation}
\rightarrow
\text{verification scheduling}
\rightarrow
\text{hardware-aware serving}
\rightarrow
\text{adaptive AI infrastructure}
\]

The central shift is:

> **Verification is not a fixed cost. Verification is a schedulable engineering resource.**


In [ ]:
from pathlib import Path
import json
import math
import os
import shutil
import sys
import textwrap
import zipfile
from dataclasses import dataclass, asdict

import matplotlib.pyplot as plt
import pandas as pd

# Make notebook paths robust whether run from repo root, notebooks/, or an uploaded copy.
CWD = Path.cwd().resolve()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "00_context"
SITE = REPO_ROOT / "site"
SITE_REPORTS = SITE / "reports" / "00"

for path in [NOTEBOOKS, FIGURES, RESULTS, SITE_REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repo root: {REPO_ROOT}")
print(f"Notebooks: {NOTEBOOKS}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")
print(f"Site:      {SITE}")


## 2. Context

Autoregressive decoding generates one token per target-model forward pass. Speculative decoding accelerates generation by using a smaller draft model to propose a block of candidate tokens, then asking the target model to verify that block in parallel.

The latency equation is:

\[
L = \frac{T_{\mathrm{draft}} + T_{\mathrm{verify}}}{\tau}
\]

where:

- \(T_{\mathrm{draft}}\) is draft-model time.
- \(T_{\mathrm{verify}}\) is target-model verification time.
- \(\tau\) is the number of accepted tokens per decoding round.

The engineering levers are therefore:

\[
\text{lower latency}
= 
\text{draft faster}
+ 
\text{accept more}
+ 
\text{verify smarter}
\]


In [ ]:
@dataclass(frozen=True)
class DecodingVariables:
    draft_time: str
    verify_time: str
    accepted_tokens: str
    batch_size: str
    steps_per_second: str
    throughput: str

variables = DecodingVariables(
    draft_time="T_draft = time spent proposing candidate tokens",
    verify_time="T_verify = time spent checking candidates with the target model",
    accepted_tokens="tau = accepted tokens per decoding round",
    batch_size="B = target-model verification batch size",
    steps_per_second="SPS(B) = profiled engine steps per second at batch size B",
    throughput="Theta = expected accepted tokens times engine steps per second",
)

pd.DataFrame([asdict(variables)]).T.rename(columns={0: "meaning"})


## 3. Leading specification flow

Notebook 00 keeps the repository organized around one flow:

\[
\text{prompt}
\rightarrow
\text{draft block}
\rightarrow
\text{confidence scores}
\rightarrow
\text{scheduled prefix}
\rightarrow
\text{target verification}
\rightarrow
\text{accepted tokens}
\rightarrow
\text{throughput}
\]

Later notebooks refine one part of this flow at a time.


In [ ]:
flow_steps = [
    "Prompt",
    "Draft\nblock",
    "Confidence\nscores",
    "Scheduled\nprefix",
    "Target\nverification",
    "Accepted\ntokens",
    "Throughput",
]

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.axis("off")
y = 0.5

for i, label in enumerate(flow_steps):
    ax.text(
        i, y, label,
        ha="center", va="center",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.45", linewidth=1.2, facecolor="white"),
    )
    if i < len(flow_steps) - 1:
        ax.annotate(
            "",
            xy=(i + 0.72, y),
            xytext=(i + 0.28, y),
            arrowprops=dict(arrowstyle="->", linewidth=1.5),
        )

ax.set_xlim(-0.6, len(flow_steps) - 0.4)
ax.set_ylim(0, 1)
ax.set_title("Confidence-scheduled decoding specification flow", fontsize=14)
fig.tight_layout()

flow_fig = FIGURES / "00_specification_flow.png"
fig.savefig(flow_fig, dpi=180, bbox_inches="tight")
plt.show()
flow_fig


## 4. Three decoding regimes

Notebook 00 uses a compact toy model to distinguish three regimes:

| Regime | Verification rule | Failure mode |
|---|---|---|
| Autoregressive | verify one generated token at a time | high latency |
| Fixed speculative | verify the full draft block | rejected suffixes waste target compute |
| Confidence-scheduled | verify only the high-value prefix | depends on calibrated confidence |

The DSpark shift is to schedule verification length per request instead of treating draft length as verification length.


In [ ]:
regimes = pd.DataFrame([
    {
        "regime": "autoregressive",
        "drafted_tokens": 1,
        "verified_tokens": 1,
        "expected_accepted_tokens": 1.0,
        "statement": "one target pass per token",
    },
    {
        "regime": "fixed speculative",
        "drafted_tokens": 7,
        "verified_tokens": 7,
        "expected_accepted_tokens": 3.5,
        "statement": "long drafts can waste verification",
    },
    {
        "regime": "confidence-scheduled",
        "drafted_tokens": 7,
        "verified_tokens": 4,
        "expected_accepted_tokens": 3.3,
        "statement": "verify fewer tokens with higher expected return",
    },
])

regimes_path = RESULTS / "decoding_regimes.csv"
regimes.to_csv(regimes_path, index=False)
regimes


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(regimes["regime"], regimes["verified_tokens"])
ax.set_ylabel("Verified tokens per round")
ax.set_title("Verification load by decoding regime")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()

regime_fig = FIGURES / "00_verification_load_by_regime.png"
fig.savefig(regime_fig, dpi=180, bbox_inches="tight")
plt.show()
regime_fig


## 5. Confidence produces prefix survival probabilities

DSpark's confidence head estimates a conditional survival probability for each draft position:

\[
c_k \in (0, 1)
\]

Because speculative decoding accepts a continuous prefix, the probability that position \(j\) survives verification is the cumulative product:

\[
a_j = \prod_{i \le j} c_i
\]

The scheduler should therefore reason over prefix survival probabilities, not isolated token confidence scores.


In [ ]:
confidence = [0.98, 0.94, 0.89, 0.66, 0.42, 0.25, 0.12]
survival = []
prod = 1.0
for c in confidence:
    prod *= c
    survival.append(prod)

confidence_df = pd.DataFrame({
    "position": list(range(1, len(confidence) + 1)),
    "conditional_confidence_c_k": confidence,
    "prefix_survival_a_j": survival,
})

confidence_path = RESULTS / "confidence_survival_example.csv"
confidence_df.to_csv(confidence_path, index=False)
confidence_df


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(confidence_df["position"], confidence_df["conditional_confidence_c_k"], marker="o", label="conditional confidence")
ax.plot(confidence_df["position"], confidence_df["prefix_survival_a_j"], marker="o", label="prefix survival")
ax.set_xlabel("Draft position")
ax.set_ylabel("Probability")
ax.set_ylim(0, 1.05)
ax.set_title("Confidence becomes a prefix survival curve")
ax.legend()
fig.tight_layout()

survival_fig = FIGURES / "00_confidence_survival_curve.png"
fig.savefig(survival_fig, dpi=180, bbox_inches="tight")
plt.show()
survival_fig


## 6. Verification budget as an allocation problem

For one request, a simple confidence threshold can prune a low-confidence suffix. For many concurrent requests, the scheduling problem becomes global:

\[
\Theta = \tau \cdot \mathrm{SPS}(B)
\]

where:

- \(\Theta\) is expected system-wide throughput.
- \(\tau\) is expected accepted tokens in the verification step.
- \(\mathrm{SPS}(B)\) is the profiled engine steps per second at verification batch size \(B\).

Notebook 00 implements a small greedy scheduler that admits prefix extensions while expected throughput improves.


In [ ]:
def sps(batch_size: int, base: float = 140.0, slope: float = 2.2) -> float:
    """Toy engine profile: steps per second decreases as target batch size grows."""
    return base / (1.0 + slope * math.log1p(batch_size))


def prefix_survival(confidences):
    out = []
    p = 1.0
    for c in confidences:
        p *= c
        out.append(p)
    return out


def schedule_prefixes(confidence_by_request):
    """Greedy early-stopping scheduler inspired by hardware-aware prefix scheduling."""
    requests = list(confidence_by_request.keys())
    R = len(requests)
    lengths = {r: 0 for r in requests}
    selected = dict(lengths)

    candidates = []
    for r in requests:
        for j, a in enumerate(prefix_survival(confidence_by_request[r]), start=1):
            candidates.append((r, j, a))
    candidates.sort(key=lambda item: item[2], reverse=True)

    B = R
    tau = R  # one target-generated bonus token per active request in this toy model
    best_theta = tau * sps(B)
    path = [{"step": 0, "request": None, "position": None, "survival": None, "B": B, "tau": tau, "theta": best_theta, "accepted": True}]

    for step, (r, j, a) in enumerate(candidates, start=1):
        if j != lengths[r] + 1:
            # Prefix constraint: cannot admit position j before j-1 for the same request.
            continue
        trial_lengths = dict(lengths)
        trial_lengths[r] = j
        trial_B = B + 1
        trial_tau = tau + a
        theta = trial_tau * sps(trial_B)
        accepted = theta > best_theta
        path.append({"step": step, "request": r, "position": j, "survival": a, "B": trial_B, "tau": trial_tau, "theta": theta, "accepted": accepted})
        if accepted:
            lengths = trial_lengths
            selected = dict(lengths)
            B = trial_B
            tau = trial_tau
            best_theta = theta
        else:
            break

    return selected, pd.DataFrame(path)

confidence_by_request = {
    "chat_A": [0.92, 0.71, 0.45, 0.22, 0.10],
    "code_B": [0.99, 0.96, 0.91, 0.84, 0.75],
    "math_C": [0.98, 0.94, 0.88, 0.74, 0.55],
    "chat_D": [0.89, 0.64, 0.33, 0.17, 0.08],
}

selected_lengths, scheduler_path = schedule_prefixes(confidence_by_request)

selected_path = RESULTS / "scheduled_prefix_lengths.json"
path_csv = RESULTS / "scheduler_path.csv"
selected_path.write_text(json.dumps(selected_lengths, indent=2), encoding="utf-8")
scheduler_path.to_csv(path_csv, index=False)

selected_lengths, scheduler_path


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
for request, scores in confidence_by_request.items():
    ax.plot(range(1, len(scores) + 1), prefix_survival(scores), marker="o", label=request)

for request, length in selected_lengths.items():
    if length > 0:
        ax.scatter([length], [prefix_survival(confidence_by_request[request])[length - 1]], s=90)

ax.set_xlabel("Draft position")
ax.set_ylabel("Prefix survival probability")
ax.set_ylim(0, 1.05)
ax.set_title("Scheduled prefixes under a toy verification budget")
ax.legend()
fig.tight_layout()

scheduled_fig = FIGURES / "00_scheduled_prefixes.png"
fig.savefig(scheduled_fig, dpi=180, bbox_inches="tight")
plt.show()
scheduled_fig


## 7. Verification budget diagram

The core idea can be drawn as a budget allocation:

\[
\text{draft block}
\rightarrow
\text{confidence scores}
\rightarrow
\text{verification budget}
\rightarrow
\text{scheduled prefix}
\]

A low-confidence suffix is not necessarily wrong. It is lower expected return under current engine load.


In [ ]:
example_tokens = ["E", "F", "G", "H", "I", "J", "K"]
example_conf = [0.98, 0.94, 0.89, 0.66, 0.42, 0.25, 0.12]
verify_length = 4

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.axis("off")
ax.set_title("Verification budget: allocate to the highest-value prefix", fontsize=15)

for i, (tok, conf) in enumerate(zip(example_tokens, example_conf)):
    label = f"{tok}\n{conf:.2f}"
    alpha = 1.0 if i < verify_length else 0.25
    ax.text(
        i, 0.62, label,
        ha="center", va="center", fontsize=12, alpha=alpha,
        bbox=dict(boxstyle="round,pad=0.45", linewidth=1.2, facecolor="white"),
    )

ax.text(1.5, 0.17, "scheduled verification prefix", ha="center", va="center", fontsize=11)
ax.annotate("", xy=(3.45, 0.35), xytext=(-0.45, 0.35), arrowprops=dict(arrowstyle="|-|", linewidth=1.8))
ax.text(5.2, 0.17, "deferred suffix", ha="center", va="center", fontsize=11)
ax.annotate("", xy=(6.45, 0.35), xytext=(4.55, 0.35), arrowprops=dict(arrowstyle="|-|", linewidth=1.2))

ax.set_xlim(-0.8, len(example_tokens) - 0.2)
ax.set_ylim(0, 1)
fig.tight_layout()

budget_fig = FIGURES / "00_verification_budget.png"
fig.savefig(budget_fig, dpi=180, bbox_inches="tight")
plt.show()
budget_fig


## 8. Throughput curve

The scheduler does not maximize accepted tokens alone. It maximizes accepted tokens under an engine profile.

Adding more verification tokens can increase expected accepts while reducing engine steps per second. The useful operating point is the best product:

\[
\Theta(B) = \tau(B)\,\mathrm{SPS}(B)
\]


In [ ]:
throughput_rows = []
R = len(confidence_by_request)
all_survivals = []
for request, scores in confidence_by_request.items():
    all_survivals.extend(prefix_survival(scores))
all_survivals = sorted(all_survivals, reverse=True)

for added in range(0, len(all_survivals) + 1):
    B = R + added
    tau = R + sum(all_survivals[:added])
    throughput_rows.append({
        "added_verification_tokens": added,
        "target_batch_size_B": B,
        "expected_accepts_tau": tau,
        "steps_per_second_SPS": sps(B),
        "throughput_theta": tau * sps(B),
    })

throughput_df = pd.DataFrame(throughput_rows)
throughput_path = RESULTS / "throughput_curve.csv"
throughput_df.to_csv(throughput_path, index=False)
throughput_df.head()


In [ ]:
best_idx = throughput_df["throughput_theta"].idxmax()
best_row = throughput_df.loc[best_idx]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(throughput_df["added_verification_tokens"], throughput_df["throughput_theta"], marker="o")
ax.scatter([best_row["added_verification_tokens"]], [best_row["throughput_theta"]], s=100)
ax.set_xlabel("Added verification tokens")
ax.set_ylabel("Expected throughput Θ")
ax.set_title("Toy throughput objective: accepted tokens × steps/sec")
fig.tight_layout()

throughput_fig = FIGURES / "00_throughput_objective.png"
fig.savefig(throughput_fig, dpi=180, bbox_inches="tight")
plt.show()
best_row.to_dict(), throughput_fig


## 9. Engineering specification summary

Notebook 00 separates the repo into four reusable specifications.

| Engineering property | Leading specification |
|---|---|
| Draft quality | semi-autoregressive generation |
| Verification value | calibrated confidence |
| Batch allocation | hardware-aware prefix scheduling |
| Serving performance | throughput objective \(\Theta = \tau \cdot \mathrm{SPS}(B)\) |

The central repository claim is:

> **Confidence estimates become scheduling signals that allocate verification compute where it produces the greatest throughput.**


In [ ]:
spec_summary = pd.DataFrame([
    {"engineering_property": "draft quality", "leading_specification": "semi-autoregressive generation", "notebook": "17_semi_autoregression"},
    {"engineering_property": "verification value", "leading_specification": "calibrated confidence", "notebook": "13_confidence_scheduling"},
    {"engineering_property": "batch allocation", "leading_specification": "hardware-aware prefix scheduling", "notebook": "29_hardware_aware_scheduling"},
    {"engineering_property": "serving performance", "leading_specification": "throughput objective", "notebook": "23_throughput_objective"},
])
summary_path = RESULTS / "engineering_specification_summary.csv"
spec_summary.to_csv(summary_path, index=False)
spec_summary


## 10. Notebook roadmap

The repo can now grow in small, testable steps.


In [ ]:
roadmap = pd.DataFrame([
    {"notebook": "00_context", "specification": "verification as resource", "question": "Why does confidence scheduling matter?"},
    {"notebook": "07_verification_resource", "specification": "verification budget", "question": "When does fixed verification waste target compute?"},
    {"notebook": "13_confidence_scheduling", "specification": "confidence-guided prefixes", "question": "How do confidence scores choose verification length?"},
    {"notebook": "17_semi_autoregression", "specification": "draft dependency", "question": "How does a lightweight sequential head reduce suffix decay?"},
    {"notebook": "23_throughput_objective", "specification": "accepted tokens times steps/sec", "question": "What objective should the scheduler maximize?"},
    {"notebook": "29_hardware_aware_scheduling", "specification": "engine-aware allocation", "question": "How does the scheduler adapt to system load?"},
    {"notebook": "37_operating_regimes", "specification": "load-dependent policy", "question": "When should the system verify more or less?"},
    {"notebook": "43_resource_allocation", "specification": "confidence as allocation signal", "question": "How does this generalize beyond one decoder?"},
    {"notebook": "49_adaptive_ai_infrastructure", "specification": "adaptive serving systems", "question": "How do confidence schedules point toward AI operating systems?"},
])
roadmap_path = RESULTS / "notebook_roadmap.csv"
roadmap.to_csv(roadmap_path, index=False)
roadmap


## 11. Repository resources

**Paper**

- DSpark: *Confidence-Scheduled Speculative Decoding with Semi-Autoregressive Generation*
- arXiv: `2606.19348v1`

**Related implementation resources**

- DeepSpec: `https://github.com/deepseek-ai/DeepSpec`
- DeepSeek-V4-Pro-DSpark model card: `https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro-DSpark`

**Lab report**

- `labreports.app/2606-19348/`
- **Confidence Scheduling Specifies Scalable LLM Serving**


In [ ]:
references = {
    "paper": {
        "identifier": "2606.19348v1",
        "title": "DSpark: Confidence-Scheduled Speculative Decoding with Semi-Autoregressive Generation",
        "url": "https://arxiv.org/html/2606.19348v1",
    },
    "implementation": {
        "deepspec_github": "https://github.com/deepseek-ai/DeepSpec",
        "model_card": "https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro-DSpark",
    },
    "lab_report": {
        "path": "/2606-19348/",
        "title": "Confidence Scheduling Specifies Scalable LLM Serving",
    },
    "repository": {
        "name": "confidence-scheduled-decoding",
        "url": "https://github.com/thinkthoughts/confidence-scheduled-decoding",
    },
}

references_path = RESULTS / "references.json"
references_path.write_text(json.dumps(references, indent=2), encoding="utf-8")
references


## 12. Export a site stub for `00.html`

This cell writes a static notebook page that can be deployed or replaced with the full labreports template later.


In [ ]:
site_html = """<!doctype html>
<html lang=\"en\" data-theme=\"light\">
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <title>00 Context | Confidence-Scheduled Decoding</title>
  <meta name=\"description\" content=\"Notebook 00 introduces confidence-scheduled decoding: verification becomes a schedulable engineering resource for scalable LLM serving.\">
  <link rel=\"stylesheet\" href=\"/styles/site.css\">
  <script>
    window.MathJax = {
      tex: { inlineMath: [[\"\\\\(\", \"\\\\)\"], [\"$\", \"$\"]] },
      svg: { fontCache: \"global\" }
    };
  </script>
  <script defer src=\"https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-svg.js\"></script>
  <script defer src=\"/js/dark-mode.js\"></script>
</head>
<body>
  <header class=\"site-header\">
    <nav class=\"site-nav\">
      <a class=\"site-brand\" href=\"/\">labreports.app</a>
      <div class=\"nav-links\">
        <a href=\"/\">Reports</a>
        <a href=\"/2606-19348/\">Lab Report</a>
        <a href=\"https://github.com/thinkthoughts/confidence-scheduled-decoding\">GitHub</a>
      </div>
      <button id=\"theme-toggle\" class=\"theme-toggle\" type=\"button\" aria-label=\"Toggle dark mode\">🌙</button>
    </nav>
  </header>
  <main class=\"site-main\">
    <article class=\"lab-report\">
      <section class=\"lab-hero\">
        <p class=\"lab-kicker\">confidence-scheduled-decoding · notebook 00</p>
        <h1>Context: Verification Becomes a Resource</h1>
        <p class=\"lab-subtitle\">Confidence scheduling specifies scalable LLM serving by treating target-model verification as a schedulable engineering resource.</p>
      </section>
      <section class=\"lab-card lab-figure-card\">
        <figure class=\"lab-figure\">
          <img src=\"../../../figures/00_specification_flow.png\" alt=\"Flow diagram from prompt to draft block to confidence scores to scheduled prefix to target verification to accepted tokens to throughput.\">
          <figcaption class=\"lab-caption\">Notebook 00 defines the repository flow: draft, estimate confidence, schedule verification, and maximize useful throughput.</figcaption>
        </figure>
      </section>
      <section class=\"lab-note\">
        <h2>Core equations</h2>
        <p>Latency: \\(L = (T_{draft} + T_{verify}) / \\tau\\)</p>
        <p>Prefix survival: \\(a_j = \\prod_{i \\le j} c_i\\)</p>
        <p>Throughput: \\(\\Theta = \\tau \\cdot SPS(B)\\)</p>
      </section>
      <section class=\"lab-card lab-figure-card\">
        <figure class=\"lab-figure\">
          <img src=\"../../../figures/00_verification_budget.png\" alt=\"Verification budget diagram showing high-confidence tokens selected as a scheduled prefix and low-confidence suffix tokens deferred.\">
          <figcaption class=\"lab-caption\">Verification budget: confidence selects the high-value prefix instead of verifying every drafted token.</figcaption>
        </figure>
      </section>
    </article>
  </main>
</body>
</html>
"""

site_path = SITE_REPORTS / "00.html"
site_path.write_text(site_html, encoding="utf-8")
site_path


## 13. Save notebook manifest

This manifest records the artifacts produced by notebook 00.


In [ ]:
notebook_manifest = {
    "notebook": "00_context.ipynb",
    "title": "Context: Verification Becomes a Resource",
    "primary_specification": "confidence scheduling specifies scalable LLM serving",
    "statement": "Verification is not a fixed cost; verification is a schedulable engineering resource.",
    "outputs": {
        "decoding_regimes_csv": str(regimes_path.relative_to(REPO_ROOT)),
        "confidence_survival_csv": str(confidence_path.relative_to(REPO_ROOT)),
        "scheduled_prefix_lengths": str(selected_path.relative_to(REPO_ROOT)),
        "scheduler_path_csv": str(path_csv.relative_to(REPO_ROOT)),
        "throughput_curve_csv": str(throughput_path.relative_to(REPO_ROOT)),
        "engineering_summary_csv": str(summary_path.relative_to(REPO_ROOT)),
        "roadmap_csv": str(roadmap_path.relative_to(REPO_ROOT)),
        "references_json": str(references_path.relative_to(REPO_ROOT)),
        "specification_flow_figure": str(flow_fig.relative_to(REPO_ROOT)),
        "verification_load_figure": str(regime_fig.relative_to(REPO_ROOT)),
        "survival_curve_figure": str(survival_fig.relative_to(REPO_ROOT)),
        "scheduled_prefixes_figure": str(scheduled_fig.relative_to(REPO_ROOT)),
        "verification_budget_figure": str(budget_fig.relative_to(REPO_ROOT)),
        "throughput_objective_figure": str(throughput_fig.relative_to(REPO_ROOT)),
        "site_stub": str(site_path.relative_to(REPO_ROOT)),
    },
}

notebook_manifest_path = RESULTS / "00_context_manifest.json"
notebook_manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")
notebook_manifest


## 14. Download artifacts

Run the final cell to package the notebook outputs for download.


In [ ]:
# Package notebook artifacts for download
from pathlib import Path
from IPython.display import FileLink, display
import zipfile

zip_path = RESULTS / "00_context_artifacts.zip"

notebook_path = NOTEBOOKS / "00_context.ipynb"
fallback_notebook_path = Path.cwd() / "00_context.ipynb"

paths_to_include = [
    notebook_path if notebook_path.exists() else fallback_notebook_path,
    FIGURES,
    RESULTS,
    SITE,
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            if item.resolve() == zip_path.resolve():
                continue
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
